# Reto II - Proyecto final 2026

Modelos avanzados.


## 1. Configuracion inicial y revision del entorno

Carga de librerias base, revision de archivos disponibles en Kaggle, configuracion de GPU y comprobacion de versiones.


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


/kaggle/input/datasets/rosalirodriguez/testgpu-2/sample_submission.csv
/kaggle/input/datasets/rosalirodriguez/testgpu-2/test_nolabel.csv
/kaggle/input/datasets/rosalirodriguez/testgpu-2/train_preprocessed.csv
/kaggle/input/datasets/rosalirodriguez/testgpu-2/train.csv


### 1.1 Instalacion opcional de dependencias


In [2]:
!pip install -q --no-cache-dir \
    torch==2.3.0 \
    torchvision==0.18.0 \
    torchaudio==2.3.0 \
    transformers==4.45.1 \
    datasets==2.21.0 \
    evaluate==0.4.3 \
    accelerate \
    sentencepiece \
    fsspec==2024.6.1 \
    gcsfs==2024.6.1 \
    "protobuf<5"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 195.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 69.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 157.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 32.8 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 53.0 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 296.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 285.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 357.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 267.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 191.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 254.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [4]:
import torch
import transformers
import datasets
import pandas as pd

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("cuda:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Sin GPU")


torch: 2.3.0+cu121
transformers: 4.45.1
datasets: 2.21.0
cuda: True
Tesla T4


## 2. Modelo clasico: Linear SVM

Baseline clasico con limpieza ligera, TF-IDF, variables categoricas y calibracion de probabilidades para ajuste de threshold.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score

SEED = 42

TRAIN_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/train.csv")
TEST_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/test_nolabel.csv")
SAMPLE_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/sample_submission.csv")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

test_ids = test_df["id"].copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\d+", " number ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def keep_top_train_test(train_series, test_series, top_n):
    train_series = train_series.fillna("missing").astype(str).str.lower().str.strip()
    test_series = test_series.fillna("missing").astype(str).str.lower().str.strip()

    top_values = set(train_series.value_counts().head(top_n).index)

    train_top = train_series.apply(lambda x: x if x in top_values else "other")
    test_top = test_series.apply(lambda x: x if x in top_values else "other")

    return train_top, test_top

def add_subject_features(train_df, test_df):
    train_subjects = (
        train_df["subject"]
        .fillna("missing")
        .astype(str)
        .str.lower()
        .str.split(",")
        .apply(lambda xs: [x.strip() for x in xs])
    )

    test_subjects = (
        test_df["subject"]
        .fillna("missing")
        .astype(str)
        .str.lower()
        .str.split(",")
        .apply(lambda xs: [x.strip() for x in xs])
    )

    mlb = MultiLabelBinarizer()

    train_encoded = mlb.fit_transform(train_subjects)
    test_encoded = mlb.transform(test_subjects)

    subject_cols = [f"subject_{s}" for s in mlb.classes_]

    train_subject_df = pd.DataFrame(train_encoded, columns=subject_cols, index=train_df.index)
    test_subject_df = pd.DataFrame(test_encoded, columns=subject_cols, index=test_df.index)

    train_df = pd.concat([train_df, train_subject_df], axis=1)
    test_df = pd.concat([test_df, test_subject_df], axis=1)

    return train_df, test_df, subject_cols

train_df["statement_clean"] = train_df["statement"].apply(clean_text)
test_df["statement_clean"] = test_df["statement"].apply(clean_text)

train_df["speaker_job_missing"] = train_df["speaker_job"].isnull().astype(int)
train_df["state_info_missing"] = train_df["state_info"].isnull().astype(int)

test_df["speaker_job_missing"] = test_df["speaker_job"].isnull().astype(int)
test_df["state_info_missing"] = test_df["state_info"].isnull().astype(int)

train_df["speaker_top"], test_df["speaker_top"] = keep_top_train_test(
    train_df["speaker"], test_df["speaker"], top_n=50
)

train_df["speaker_job_top"], test_df["speaker_job_top"] = keep_top_train_test(
    train_df["speaker_job"], test_df["speaker_job"], top_n=50
)

train_df["state_info_top"], test_df["state_info_top"] = keep_top_train_test(
    train_df["state_info"], test_df["state_info"], top_n=30
)

TOP_PARTIES = {"republican", "democrat", "none", "organization", "independent"}

train_party = train_df["party_affiliation"].fillna("missing").astype(str).str.lower().str.strip()
test_party = test_df["party_affiliation"].fillna("missing").astype(str).str.lower().str.strip()

train_df["party_top"] = train_party.apply(lambda x: x if x in TOP_PARTIES else "other")
test_df["party_top"] = test_party.apply(lambda x: x if x in TOP_PARTIES else "other")

train_df, test_df, subject_cols = add_subject_features(train_df, test_df)

feature_cols = [
    "statement_clean",
    "speaker_top",
    "speaker_job_top",
    "state_info_top",
    "party_top",
    "speaker_job_missing",
    "state_info_missing",
] + subject_cols

X = train_df[feature_cols].copy()
y = train_df["label"].copy()
X_test_kaggle = test_df[feature_cols].copy()

tr_idx, val_idx = train_test_split(
    train_df.index,
    test_size=0.2,
    stratify=y,
    random_state=SEED,
)

X_train = X.loc[tr_idx]
X_val = X.loc[val_idx]
y_train = y.loc[tr_idx]
y_val = y.loc[val_idx]

cat_cols = ["speaker_top", "speaker_job_top", "state_info_top", "party_top"]
num_cols = ["speaker_job_missing", "state_info_missing"] + subject_cols

preprocessor = ColumnTransformer(
    transformers=[
        (
            "word_tfidf",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 3),
                min_df=2,
                max_df=0.90,
                max_features=30000,
                sublinear_tf=True,
            ),
            "statement_clean",
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_cols,
        ),
        (
            "num",
            "passthrough",
            num_cols,
        ),
    ]
)

svm_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LinearSVC(
            C=0.1,
            loss="squared_hinge",
            class_weight="balanced",
            max_iter=7000,
        )),
    ]
)

calibrated_svm = CalibratedClassifierCV(
    estimator=svm_pipeline,
    method="sigmoid",
    cv=3,
)

calibrated_svm.fit(X_train, y_train)

svm_val_probs = calibrated_svm.predict_proba(X_val)[:, 1]

svm_val_df = pd.DataFrame({
    "id": train_df.loc[val_idx, "id"].values,
    "label": y_val.values,
    "svm_proba_1": svm_val_probs,
})

calibrated_svm.fit(X, y)

svm_test_probs = calibrated_svm.predict_proba(X_test_kaggle)[:, 1]

svm_test_df = pd.DataFrame({
    "id": test_ids,
    "svm_proba_1": svm_test_probs,
})

roberta_val = pd.read_csv("/kaggle/working/roberta_val_probs.csv")
roberta_test = pd.read_csv("/kaggle/working/roberta_test_probs.csv")

val_ens = roberta_val.merge(svm_val_df, on=["id", "label"])
test_ens = roberta_test.merge(svm_test_df, on="id")

best_score = -1
best_weight = None
best_threshold = None

for weight_roberta in np.arange(0.0, 1.01, 0.05):
    val_proba = (
        weight_roberta * val_ens["roberta_proba_1"]
        + (1 - weight_roberta) * val_ens["svm_proba_1"]
    )

    for threshold in np.arange(0.30, 0.71, 0.01):
        preds = (val_proba >= threshold).astype(int)
        score = f1_score(val_ens["label"], preds, average="macro")

        if score > best_score:
            best_score = score
            best_weight = weight_roberta
            best_threshold = threshold

print("Best validation ensemble F1:", best_score)
print("Best RoBERTa weight:", best_weight)
print("Best threshold:", best_threshold)

for threshold in [
    best_threshold - 0.03,
    best_threshold - 0.01,
    best_threshold,
    best_threshold + 0.01,
    best_threshold + 0.03,
]:
    threshold = round(float(threshold), 2)

    test_proba = (
        best_weight * test_ens["roberta_proba_1"]
        + (1 - best_weight) * test_ens["svm_proba_1"]
    )

    test_preds = (test_proba >= threshold).astype(int)

    submission = sample_submission.copy()
    submission["label"] = submission["id"].map(dict(zip(test_ens["id"], test_preds)))

    output_path = Path("/kaggle/working") / f"submission_ensemble_roberta_svm_t{threshold:.2f}.csv"
    submission.to_csv(output_path, index=False)

    print("\nThreshold:", threshold)
    print("Archivo:", output_path)
    print(submission["label"].value_counts())


## 3. Transformer: RoBERTa-large

Mejor prueba individual del proyecto. Usa `model_text`, combinando statement y metadatos contextuales. Genera submissions con distintos thresholds y guarda probabilidades para posibles ensembles.


In [6]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from scipy.special import softmax

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/train.csv")
TEST_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/test_nolabel.csv")
SAMPLE_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/sample_submission.csv")

#MODEL_NAME = "microsoft/deberta-v3-base"
#MODEL_NAME = "roberta-base"
MODEL_NAME = "FacebookAI/roberta-large"
MAX_LEN = 256

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Sin GPU")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

test_ids = test_df["id"].copy()

def build_model_text(df):
    df = df.copy()

    cols = [
        "statement",
        "subject",
        "speaker",
        "speaker_job",
        "state_info",
        "party_affiliation",
    ]

    for col in cols:
        df[col] = df[col].fillna("unknown").astype(str).str.lower().str.strip()

    df["model_text"] = (
        "statement: " + df["statement"]
        + " subject: " + df["subject"]
        + " speaker: " + df["speaker"]
        + " job: " + df["speaker_job"]
        + " state: " + df["state_info"]
        + " party: " + df["party_affiliation"]
    )

    return df

train_df = build_model_text(train_df)
test_df = build_model_text(test_df)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print(train_df["label"].value_counts())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def encode(batch):
    return tokenizer(
        batch["model_text"],
        truncation=True,
        max_length=MAX_LEN,
    )

tr_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=SEED,
)

train_ds = Dataset.from_pandas(tr_df[["model_text", "label"]])
val_ds = Dataset.from_pandas(val_df[["model_text", "label"]])
test_ds = Dataset.from_pandas(test_df[["model_text"]])

train_ds = train_ds.map(encode, batched=True, remove_columns=["model_text"])
val_ds = val_ds.map(encode, batched=True, remove_columns=["model_text"])
test_ds = test_ds.map(encode, batched=True, remove_columns=["model_text"])

collator = DataCollatorWithPadding(
    tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "f1": f1_score(labels, preds, average="macro")
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)

args = TrainingArguments(
    output_dir="/kaggle/working/roberta_large_metadata",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    warmup_ratio=0.1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    weight_decay=0.01,
    max_grad_norm=0.5,
    fp16=False,
    bf16=False,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=0,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

metrics = trainer.evaluate()
print("F1 macro valid argmax:", metrics["eval_f1"])

val_logits = trainer.predict(val_ds).predictions
val_probs = softmax(val_logits, axis=1)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.01)
scores = []

for threshold in thresholds:
    val_preds = (val_probs >= threshold).astype(int)
    scores.append(f1_score(val_df["label"].values, val_preds, average="macro"))

best_idx = np.argmax(scores)
best_threshold = thresholds[best_idx]

print("Mejor threshold:", best_threshold)
print("Mejor F1 macro valid:", scores[best_idx])

test_logits = trainer.predict(test_ds).predictions
test_probs = softmax(test_logits, axis=1)[:, 1]

thresholds_to_submit = [
    best_threshold - 0.04,
    best_threshold - 0.02,
    best_threshold,
    best_threshold + 0.02,
    best_threshold + 0.04,
]

thresholds_to_submit = [
    round(float(t), 2)
    for t in thresholds_to_submit
    if 0.05 <= t <= 0.95
]

#for threshold in thresholds_to_submit:
#for threshold in [0.57, 0.61, 0.63, 0.65]:
#for threshold in [0.56, 0.57, 0.58, 0.59, 0.60]:
for threshold in [0.54, 0.56, 0.58, 0.59, 0.60, 0.62]:    
    preds = (test_probs >= threshold).astype(int)

    submission = sample_submission.copy()
    submission["label"] = submission["id"].map(dict(zip(test_ids, preds)))

    output_path = Path("/kaggle/working") / f"submission_roberta_base_metadata_t{threshold:.2f}.csv"
    submission.to_csv(output_path, index=False)

    print("\nThreshold:", threshold)
    print("Archivo:", output_path)
    print(submission["label"].value_counts())

argmax_preds = np.argmax(test_logits, axis=-1)

submission_argmax = sample_submission.copy()
submission_argmax["label"] = submission_argmax["id"].map(dict(zip(test_ids, argmax_preds)))

argmax_path = Path("/kaggle/working/submission_roberta_base_metadatax.csv")
submission_argmax.to_csv(argmax_path, index=False)

print("\nArchivo argmax:", argmax_path)
print(submission_argmax["label"].value_counts())


CUDA: True
GPU: Tesla T4
Train: (8950, 9)
Test: (3836, 8)
label
1    5795
0    3155
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/7160 [00:00<?, ? examples/s]

Map:   0%|          | 0/1790 [00:00<?, ? examples/s]

Map:   0%|          | 0/3836 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1
0,No log,0.618532,0.557922
2,0.616500,0.592401,0.640852


F1 macro valid argmax: 0.6408521252527104
Mejor threshold: 0.5599999999999997
Mejor F1 macro valid: 0.6533535762483131



Threshold: 0.54
Archivo: /kaggle/working/submission_roberta_base_metadata_t0.54.csv
label
1    2565
0    1271
Name: count, dtype: int64

Threshold: 0.56
Archivo: /kaggle/working/submission_roberta_base_metadata_t0.56.csv
label
1    2477
0    1359
Name: count, dtype: int64

Threshold: 0.58
Archivo: /kaggle/working/submission_roberta_base_metadata_t0.58.csv
label
1    2367
0    1469
Name: count, dtype: int64

Threshold: 0.59
Archivo: /kaggle/working/submission_roberta_base_metadata_t0.59.csv
label
1    2309
0    1527
Name: count, dtype: int64

Threshold: 0.6
Archivo: /kaggle/working/submission_roberta_base_metadata_t0.60.csv
label
1    2265
0    1571
Name: count, dtype: int64

Threshold: 0.62
Archivo: /kaggle/working/submission_roberta_base_metadata_t0.62.csv
label
1    2187
0    1649
Name: count, dtype: int64

Archivo argmax: /kaggle/working/submission_roberta_base_metadatax.csv
label
1    2759
0    1077
Name: count, dtype: int64


## 4. Transformer: ELECTRA-large

Prueba con ELECTRA-large usando la misma estrategia de `model_text`. Sirve como comparacion directa con RoBERTa y como candidato para ensemble.


In [7]:
from pathlib import Path
import random
import gc

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from scipy.special import softmax

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/train.csv")
TEST_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/test_nolabel.csv")
SAMPLE_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/sample_submission.csv")

MODEL_NAME = "google/electra-large-discriminator"
MAX_LEN = 256

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Sin GPU")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

test_ids = test_df["id"].copy()

def build_model_text(df):
    df = df.copy()

    cols = [
        "statement",
        "subject",
        "speaker",
        "speaker_job",
        "state_info",
        "party_affiliation",
    ]

    for col in cols:
        df[col] = df[col].fillna("unknown").astype(str).str.lower().str.strip()

    df["model_text"] = (
        "statement: " + df["statement"]
        + " subject: " + df["subject"]
        + " speaker: " + df["speaker"]
        + " job: " + df["speaker_job"]
        + " state: " + df["state_info"]
        + " party: " + df["party_affiliation"]
    )

    return df

train_df = build_model_text(train_df)
test_df = build_model_text(test_df)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print(train_df["label"].value_counts())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def encode(batch):
    return tokenizer(
        batch["model_text"],
        truncation=True,
        max_length=MAX_LEN,
    )

tr_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=SEED,
)

train_ds = Dataset.from_pandas(tr_df[["model_text", "label"]])
val_ds = Dataset.from_pandas(val_df[["model_text", "label"]])
test_ds = Dataset.from_pandas(test_df[["model_text"]])

train_ds = train_ds.map(encode, batched=True, remove_columns=["model_text"])
val_ds = val_ds.map(encode, batched=True, remove_columns=["model_text"])
test_ds = test_ds.map(encode, batched=True, remove_columns=["model_text"])

collator = DataCollatorWithPadding(
    tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, preds, average="macro")}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)

args = TrainingArguments(
    output_dir="/kaggle/working/electra_large_metadata",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    warmup_ratio=0.1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    weight_decay=0.01,
    max_grad_norm=0.5,
    fp16=False,
    bf16=False,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=0,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

metrics = trainer.evaluate()
print("F1 macro valid argmax:", metrics["eval_f1"])

val_logits = trainer.predict(val_ds).predictions
val_probs = softmax(val_logits, axis=1)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.01)
scores = []

for threshold in thresholds:
    val_preds = (val_probs >= threshold).astype(int)
    scores.append(f1_score(val_df["label"].values, val_preds, average="macro"))

best_idx = np.argmax(scores)
best_threshold = thresholds[best_idx]
best_f1 = scores[best_idx]

print("Mejor threshold:", best_threshold)
print("Mejor F1 macro valid:", best_f1)

test_logits = trainer.predict(test_ds).predictions
test_probs = softmax(test_logits, axis=1)[:, 1]

# Guardar probabilidades para posible ensemble posterior
electra_val_probs = pd.DataFrame({
    "id": val_df["id"].values,
    "label": val_df["label"].values,
    "electra_proba_1": val_probs,
})

electra_test_probs = pd.DataFrame({
    "id": test_ids,
    "electra_proba_1": test_probs,
})

electra_val_probs.to_csv("/kaggle/working/electra_val_probs.csv", index=False)
electra_test_probs.to_csv("/kaggle/working/electra_test_probs.csv", index=False)

thresholds_to_submit = [
    best_threshold - 0.04,
    best_threshold - 0.02,
    best_threshold,
    best_threshold + 0.02,
    best_threshold + 0.04,
    0.54,
    0.56,
    0.58,
    0.59,
    0.60,
]

thresholds_to_submit = sorted(set([
    round(float(t), 2)
    for t in thresholds_to_submit
    if 0.05 <= t <= 0.95
]))

for threshold in thresholds_to_submit:
    preds = (test_probs >= threshold).astype(int)

    submission = sample_submission.copy()
    submission["label"] = submission["id"].map(dict(zip(test_ids, preds)))

    output_path = Path("/kaggle/working") / f"submission_electra_large_metadata_t{threshold:.2f}.csv"
    submission.to_csv(output_path, index=False)

    print("\nThreshold:", threshold)
    print("Archivo:", output_path)
    print(submission["label"].value_counts())

argmax_preds = np.argmax(test_logits, axis=-1)

submission_argmax = sample_submission.copy()
submission_argmax["label"] = submission_argmax["id"].map(dict(zip(test_ids, argmax_preds)))

argmax_path = Path("/kaggle/working/submission_electra_large_metadata_argmax.csv")
submission_argmax.to_csv(argmax_path, index=False)

print("\nArchivo argmax:", argmax_path)
print(submission_argmax["label"].value_counts())

gc.collect()
torch.cuda.empty_cache()


CUDA: True
GPU: Tesla T4
Train: (8950, 9)
Test: (3836, 8)
label
1    5795
0    3155
Name: count, dtype: int64


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at google/electra-large-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1
0,No log,0.604444,0.580624
2,0.552700,0.645417,0.639662


F1 macro valid argmax: 0.6396621088904477
Mejor threshold: 0.6299999999999997
Mejor F1 macro valid: 0.6484716995065234



Threshold: 0.54
Archivo: /kaggle/working/submission_electra_large_metadata_t0.54.csv
label
1    2221
0    1615
Name: count, dtype: int64

Threshold: 0.56
Archivo: /kaggle/working/submission_electra_large_metadata_t0.56.csv
label
1    2197
0    1639
Name: count, dtype: int64

Threshold: 0.58
Archivo: /kaggle/working/submission_electra_large_metadata_t0.58.csv
label
1    2145
0    1691
Name: count, dtype: int64

Threshold: 0.59
Archivo: /kaggle/working/submission_electra_large_metadata_t0.59.csv
label
1    2129
0    1707
Name: count, dtype: int64

Threshold: 0.6
Archivo: /kaggle/working/submission_electra_large_metadata_t0.60.csv
label
1    2113
0    1723
Name: count, dtype: int64

Threshold: 0.61
Archivo: /kaggle/working/submission_electra_large_metadata_t0.61.csv
label
1    2099
0    1737
Name: count, dtype: int64

Threshold: 0.63
Archivo: /kaggle/working/submission_electra_large_metadata_t0.63.csv
label
1    2064
0    1772
Name: count, dtype: int64

Threshold: 0.65
Archivo: /kaggle/w

## 5. Ensemble RoBERTa + ELECTRA

Combina probabilidades de RoBERTa y ELECTRA. Esta prueba permite comprobar si ambos modelos aportan informaci?n complementaria.


In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

ROOT = Path("/kaggle/working")
SAMPLE_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/sample_submission.csv")

# Archivos de RoBERTa 5-fold
roberta_val = pd.read_csv(ROOT / "roberta_large_5fold_oof_probs.csv")
roberta_test = pd.read_csv(ROOT / "roberta_large_5fold_test_probs.csv")

# Archivos de ELECTRA
electra_val = pd.read_csv(ROOT / "electra_val_probs.csv")
electra_test = pd.read_csv(ROOT / "electra_test_probs.csv")

print("RoBERTa val columns:", roberta_val.columns.tolist())
print("RoBERTa test columns:", roberta_test.columns.tolist())

# Ajustar nombres si vienen del 5-fold
if "roberta_large_oof_proba_1" in roberta_val.columns:
    roberta_val = roberta_val.rename(
        columns={"roberta_large_oof_proba_1": "roberta_proba_1"}
    )

if "roberta_large_test_proba_1" in roberta_test.columns:
    roberta_test = roberta_test.rename(
        columns={"roberta_large_test_proba_1": "roberta_proba_1"}
    )


if "roberta_proba_1" not in roberta_val.columns:
    proba_cols = [c for c in roberta_val.columns if "proba" in c.lower()]
    roberta_val = roberta_val.rename(columns={proba_cols[0]: "roberta_proba_1"})

if "roberta_proba_1" not in roberta_test.columns:
    proba_cols = [c for c in roberta_test.columns if "proba" in c.lower()]
    roberta_test = roberta_test.rename(columns={proba_cols[0]: "roberta_proba_1"})

# Unir validación
val = roberta_val.merge(
    electra_val,
    on=["id", "label"],
    how="inner"
)

# Unir test
test = roberta_test.merge(
    electra_test,
    on="id",
    how="inner"
)

print("Val shape:", val.shape)
print("Test shape:", test.shape)

best_score = -1
best_weight = None
best_threshold = None

for roberta_weight in np.arange(0.0, 1.01, 0.02):
    ensemble_val_proba = (
        roberta_weight * val["roberta_proba_1"]
        + (1 - roberta_weight) * val["electra_proba_1"]
    )

    for threshold in np.arange(0.40, 0.76, 0.01):
        preds = (ensemble_val_proba >= threshold).astype(int)
        score = f1_score(val["label"], preds, average="macro")

        if score > best_score:
            best_score = score
            best_weight = roberta_weight
            best_threshold = threshold

print("Best validation ensemble F1:", best_score)
print("Best RoBERTa weight:", best_weight)
print("Best ELECTRA weight:", 1 - best_weight)
print("Best threshold:", best_threshold)

sample_submission = pd.read_csv(SAMPLE_PATH)

thresholds_to_try = [
    best_threshold - 0.04,
    best_threshold - 0.02,
    best_threshold,
    best_threshold + 0.02,
    best_threshold + 0.04,
]

for threshold in thresholds_to_try:
    threshold = round(float(threshold), 2)

    ensemble_test_proba = (
        best_weight * test["roberta_proba_1"]
        + (1 - best_weight) * test["electra_proba_1"]
    )

    preds = (ensemble_test_proba >= threshold).astype(int)

    submission = sample_submission.copy()
    submission["label"] = submission["id"].map(dict(zip(test["id"], preds)))

    output_path = ROOT / f"submission_ensemble_roberta_electra_t{threshold:.2f}.csv"
    submission.to_csv(output_path, index=False)

    print("\nThreshold:", threshold)
    print("Archivo:", output_path)
    print(submission["label"].value_counts())


RoBERTa val columns: ['id', 'label', 'roberta_large_oof_proba_1']
RoBERTa test columns: ['id', 'roberta_large_test_proba_1']
Val shape: (1790, 4)
Test shape: (3836, 3)
Best validation ensemble F1: 0.6484716995065234
Best RoBERTa weight: 0.0
Best ELECTRA weight: 1.0
Best threshold: 0.6300000000000002

Threshold: 0.59
Archivo: /kaggle/working/submission_ensemble_roberta_electra_t0.59.csv
label
1    2129
0    1707
Name: count, dtype: int64

Threshold: 0.61
Archivo: /kaggle/working/submission_ensemble_roberta_electra_t0.61.csv
label
1    2099
0    1737
Name: count, dtype: int64

Threshold: 0.63
Archivo: /kaggle/working/submission_ensemble_roberta_electra_t0.63.csv
label
1    2064
0    1772
Name: count, dtype: int64

Threshold: 0.65
Archivo: /kaggle/working/submission_ensemble_roberta_electra_t0.65.csv
label
1    2022
0    1814
Name: count, dtype: int64

Threshold: 0.67
Archivo: /kaggle/working/submission_ensemble_roberta_electra_t0.67.csv
label
1    1978
0    1858
Name: count, dtype: int64

## 6. Prueba experimental: RoBERTa NLI / FEVER

Modelo preentrenado en tareas de inferencia textual y verificacion. Se incluyo como experimento para evaluar si un preentrenamiento mas cercano a claim verification podia aportar mejora.


In [11]:
from pathlib import Path
import random
import gc

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from scipy.special import softmax

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)


# CONFIG


SEEDS = [42]  # si tienes tiempo luego prueba [42, 7, 13]
MODEL_NAME = "ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli"

TRAIN_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/train.csv")
TEST_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/test_nolabel.csv")
SAMPLE_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/sample_submission.csv")

OUTPUT_ROOT = Path("/kaggle/working")
MAX_LEN = 256

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Sin GPU")
print("GPUs:", torch.cuda.device_count())


# DATA


train_df_original = pd.read_csv(TRAIN_PATH)
test_df_original = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

test_ids = test_df_original["id"].copy()

def build_model_text(df):
    df = df.copy()

    cols = [
        "statement",
        "subject",
        "speaker",
        "speaker_job",
        "state_info",
        "party_affiliation",
    ]

    for col in cols:
        df[col] = (
            df[col]
            .fillna("unknown")
            .astype(str)
            .str.strip()
        )

    # No lower-case. En transformers grandes conviene conservar nombres propios.
    df["model_text"] = (
        "Claim: " + df["statement"]
        + " Topic: " + df["subject"]
        + " Speaker: " + df["speaker"]
        + " Speaker job: " + df["speaker_job"]
        + " State: " + df["state_info"]
        + " Party: " + df["party_affiliation"]
    )

    return df

train_df_original = build_model_text(train_df_original)
test_df_original = build_model_text(test_df_original)

print("Train:", train_df_original.shape)
print("Test:", test_df_original.shape)
print(train_df_original["label"].value_counts())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def encode(batch):
    return tokenizer(
        batch["model_text"],
        truncation=True,
        max_length=MAX_LEN,
    )

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1": f1_score(labels, preds, average="macro")
    }

all_test_probs = []
all_val_results = []


# TRAIN LOOP


for seed in SEEDS:
    print("\n" + "=" * 80)
    print("SEED:", seed)
    print("=" * 80)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    train_df, val_df = train_test_split(
        train_df_original,
        test_size=0.2,
        stratify=train_df_original["label"],
        random_state=seed,
    )

    train_ds = Dataset.from_pandas(train_df[["model_text", "label"]])
    val_ds = Dataset.from_pandas(val_df[["model_text", "label"]])
    test_ds = Dataset.from_pandas(test_df_original[["model_text"]])

    train_ds = train_ds.map(encode, batched=True, remove_columns=["model_text"])
    val_ds = val_ds.map(encode, batched=True, remove_columns=["model_text"])
    test_ds = test_ds.map(encode, batched=True, remove_columns=["model_text"])

    collator = DataCollatorWithPadding(
        tokenizer,
        pad_to_multiple_of=8 if torch.cuda.is_available() else None,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        ignore_mismatched_sizes=True,
    )

    output_dir = OUTPUT_ROOT / f"nli_roberta_large_seed_{seed}"

    args_kwargs = dict(
        output_dir=str(output_dir),
        save_strategy="epoch",
        learning_rate=1e-5,
        warmup_ratio=0.1,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        weight_decay=0.01,
        max_grad_norm=0.5,
        fp16=False,
        bf16=False,
        report_to="none",
        seed=seed,
        dataloader_num_workers=0,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=1,
    )

    try:
        args = TrainingArguments(
            evaluation_strategy="epoch",
            **args_kwargs,
        )
    except TypeError:
        args = TrainingArguments(
            eval_strategy="epoch",
            **args_kwargs,
        )

    try:
        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            tokenizer=tokenizer,
            data_collator=collator,
            compute_metrics=compute_metrics,
        )
    except TypeError:
        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            tokenizer=tokenizer,
            data_collator=collator,
            compute_metrics=compute_metrics,
        )

    trainer.train()

    metrics = trainer.evaluate()
    print("F1 macro valid argmax:", metrics["eval_f1"])

    val_logits = trainer.predict(val_ds).predictions
    val_probs = softmax(val_logits, axis=1)[:, 1]

    thresholds = np.arange(0.40, 0.76, 0.01)
    scores = []

    for threshold in thresholds:
        val_preds = (val_probs >= threshold).astype(int)
        score = f1_score(val_df["label"].values, val_preds, average="macro")
        scores.append(score)

    best_idx = np.argmax(scores)
    best_threshold = float(thresholds[best_idx])
    best_f1 = float(scores[best_idx])

    print("Best threshold seed:", best_threshold)
    print("Best F1 valid threshold seed:", best_f1)

    all_val_results.append({
        "seed": seed,
        "argmax_f1": metrics["eval_f1"],
        "best_threshold": best_threshold,
        "best_threshold_f1": best_f1,
    })

    test_logits = trainer.predict(test_ds).predictions
    test_probs = softmax(test_logits, axis=1)[:, 1]

    all_test_probs.append(test_probs)

    # limpiar memoria
    del model
    del trainer
    torch.cuda.empty_cache()
    gc.collect()


# AVERAGE TEST PROBS


mean_test_probs = np.mean(all_test_probs, axis=0)

results_df = pd.DataFrame(all_val_results)
print("\nResultados por seed:")
display(results_df)

mean_best_threshold = results_df["best_threshold"].mean()
print("Threshold promedio:", mean_best_threshold)

# También probamos thresholds alrededor del mejor promedio
thresholds_to_submit = sorted(set([
    round(mean_best_threshold - 0.04, 2),
    round(mean_best_threshold - 0.02, 2),
    round(mean_best_threshold, 2),
    round(mean_best_threshold + 0.02, 2),
    round(mean_best_threshold + 0.04, 2),
    0.54,
    0.56,
    0.58,
    0.60,
    0.62,
    0.64,
]))

for threshold in thresholds_to_submit:
    if threshold < 0.05 or threshold > 0.95:
        continue

    preds = (mean_test_probs >= threshold).astype(int)

    submission = sample_submission.copy()
    submission["label"] = submission["id"].map(dict(zip(test_ids, preds)))

    output_path = OUTPUT_ROOT / f"submission_nli_roberta_large_t{threshold:.2f}.csv"
    submission.to_csv(output_path, index=False)

    print("\nThreshold:", threshold)
    print("Archivo:", output_path)
    print(submission["label"].value_counts())


CUDA: True
GPU: Tesla T4
GPUs: 1
Train: (8950, 9)
Test: (3836, 8)
label
1    5795
0    3155
Name: count, dtype: int64


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/703 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Epoch,Training Loss,Validation Loss,F1
0,No log,0.607742,0.620865
2,0.543900,0.697309,0.632954


F1 macro valid argmax: 0.6329536832205871
Best threshold seed: 0.5600000000000002
Best F1 valid threshold seed: 0.6353824852074106



Resultados por seed:


,seed,argmax_f1,best_threshold,best_threshold_f1
0,42,0.632954,0.56,0.635382


Threshold promedio: 0.5600000000000002

Threshold: 0.52
Archivo: /kaggle/working/submission_nli_roberta_large_t0.52.csv
label
1    2261
0    1575
Name: count, dtype: int64

Threshold: 0.54
Archivo: /kaggle/working/submission_nli_roberta_large_t0.54.csv
label
1    2217
0    1619
Name: count, dtype: int64

Threshold: 0.56
Archivo: /kaggle/working/submission_nli_roberta_large_t0.56.csv
label
1    2169
0    1667
Name: count, dtype: int64

Threshold: 0.58
Archivo: /kaggle/working/submission_nli_roberta_large_t0.58.csv
label
1    2115
0    1721
Name: count, dtype: int64

Threshold: 0.6
Archivo: /kaggle/working/submission_nli_roberta_large_t0.60.csv
label
1    2067
0    1769
Name: count, dtype: int64

Threshold: 0.62
Archivo: /kaggle/working/submission_nli_roberta_large_t0.62.csv
label
1    2024
0    1812
Name: count, dtype: int64

Threshold: 0.64
Archivo: /kaggle/working/submission_nli_roberta_large_t0.64.csv
label
1    1973
0    1863
Name: count, dtype: int64


## 7. Pruebas experimentales: BART-large-MNLI

BART-large-MNLI se prob? como modelo NLI alternativo. Estas celdas son experimentales y pueden requerir m?s memoria GPU que RoBERTa/ELECTRA.


### 7.1 BART-large-MNLI - versi?n inicial


In [13]:
from pathlib import Path
import random
import gc

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from scipy.special import softmax

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)


# CONFIG


SEED = 42

TRAIN_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/train.csv")
TEST_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/test_nolabel.csv")
SAMPLE_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/sample_submission.csv")

MODEL_NAME = "facebook/bart-large-mnli"
OUTPUT_DIR = Path("/kaggle/working/bart_large_mnli_metadata")
MAX_LEN = 256

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Sin GPU")
print("GPUs:", torch.cuda.device_count())


# DATA


train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

test_ids = test_df["id"].copy()

def build_model_text(df):
    df = df.copy()

    cols = [
        "statement",
        "subject",
        "speaker",
        "speaker_job",
        "state_info",
        "party_affiliation",
    ]

    for col in cols:
        df[col] = (
            df[col]
            .fillna("unknown")
            .astype(str)
            .str.strip()
        )

    df["model_text"] = (
        "Claim: " + df["statement"]
        + " Topic: " + df["subject"]
        + " Speaker: " + df["speaker"]
        + " Speaker job: " + df["speaker_job"]
        + " State: " + df["state_info"]
        + " Party: " + df["party_affiliation"]
    )

    return df

train_df = build_model_text(train_df)
test_df = build_model_text(test_df)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Label distribution:")
print(train_df["label"].value_counts())


# TOKENIZER


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def encode(batch):
    return tokenizer(
        batch["model_text"],
        truncation=True,
        max_length=MAX_LEN,
    )

tr_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=SEED,
)

train_ds = Dataset.from_pandas(tr_df[["model_text", "label"]])
val_ds = Dataset.from_pandas(val_df[["model_text", "label"]])
test_ds = Dataset.from_pandas(test_df[["model_text"]])

train_ds = train_ds.map(encode, batched=True, remove_columns=["model_text"])
val_ds = val_ds.map(encode, batched=True, remove_columns=["model_text"])
test_ds = test_ds.map(encode, batched=True, remove_columns=["model_text"])

collator = DataCollatorWithPadding(
    tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)


# HELPERS


def extract_logits(predictions):
    """
    Algunos modelos como BART pueden devolver una tupla:
    (logits, past_key_values, ...)
    Esta función extrae solo los logits.
    """
    if isinstance(predictions, tuple):
        return predictions[0]
    return predictions

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    logits = extract_logits(predictions)
    preds = np.argmax(logits, axis=-1)

    return {
        "f1": f1_score(labels, preds, average="macro")
    }


# MODEL


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)

model.config.use_cache = False


# TRAINING


args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=8e-6,
    warmup_ratio=0.1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    weight_decay=0.01,
    max_grad_norm=0.5,
    fp16=False,
    bf16=False,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=0,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
)

try:
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )
except TypeError:
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

trainer.train()

metrics = trainer.evaluate()
print("F1 macro valid argmax:", metrics["eval_f1"])


# VALIDATION THRESHOLD


val_predictions = trainer.predict(val_ds).predictions
val_logits = extract_logits(val_predictions)
val_probs = softmax(val_logits, axis=1)[:, 1]

thresholds = np.arange(0.40, 0.76, 0.01)
scores = []

for threshold in thresholds:
    val_preds = (val_probs >= threshold).astype(int)
    score = f1_score(val_df["label"].values, val_preds, average="macro")
    scores.append(score)

best_idx = np.argmax(scores)
best_threshold = float(thresholds[best_idx])
best_f1 = float(scores[best_idx])

print("Mejor threshold:", best_threshold)
print("Mejor F1 macro valid:", best_f1)


# TEST PREDICTIONS


test_predictions = trainer.predict(test_ds).predictions
test_logits = extract_logits(test_predictions)
test_probs = softmax(test_logits, axis=1)[:, 1]

# Guardar probabilidades para ensemble
bart_val_probs = pd.DataFrame({
    "id": val_df["id"].values,
    "label": val_df["label"].values,
    "bart_proba_1": val_probs,
})

bart_test_probs = pd.DataFrame({
    "id": test_ids,
    "bart_proba_1": test_probs,
})

bart_val_probs.to_csv("/kaggle/working/bart_val_probs.csv", index=False)
bart_test_probs.to_csv("/kaggle/working/bart_test_probs.csv", index=False)

print("Probabilidades guardadas:")
print("/kaggle/working/bart_val_probs.csv")
print("/kaggle/working/bart_test_probs.csv")


# SUBMISSIONS


thresholds_to_submit = sorted(set([
    round(best_threshold - 0.04, 2),
    round(best_threshold - 0.02, 2),
    round(best_threshold, 2),
    round(best_threshold + 0.02, 2),
    round(best_threshold + 0.04, 2),
    0.52,
    0.54,
    0.56,
    0.58,
    0.60,
    0.62,
    0.64,
]))

for threshold in thresholds_to_submit:
    if threshold < 0.05 or threshold > 0.95:
        continue

    preds = (test_probs >= threshold).astype(int)

    submission = sample_submission.copy()
    submission["label"] = submission["id"].map(dict(zip(test_ids, preds)))

    output_path = Path("/kaggle/working") / f"submission_bart_large_mnli_t{threshold:.2f}.csv"
    submission.to_csv(output_path, index=False)

    print("\nThreshold:", threshold)
    print("Archivo:", output_path)
    print(submission["label"].value_counts())


# CLEAN MEMORY


torch.cuda.empty_cache()
gc.collect()


CUDA: True
GPU: Tesla T4
GPUs: 1
Train: (8950, 9)
Test: (3836, 8)
Label distribution:
label
1    5795
0    3155
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/7160 [00:00<?, ? examples/s]

Map:   0%|          | 0/1790 [00:00<?, ? examples/s]

Map:   0%|          | 0/3836 [00:00<?, ? examples/s]

Some weights of BartForSequenceClassification were not initialized from the model checkpoint at facebook/bart-large-mnli and are newly initialized because the shapes did not match:
- classification_head.out_proj.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
- classification_head.out_proj.weight: found shape torch.Size([3, 1024]) in the checkpoint and torch.Size([2, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 940.00 MiB. GPU 

### 7.2 BART-large-MNLI - version con manejo de logits/memoria


In [14]:
from pathlib import Path
import random
import gc

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from scipy.special import softmax

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)


# CONFIG


SEED = 42

TRAIN_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/train.csv")
TEST_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/test_nolabel.csv")
SAMPLE_PATH = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2/sample_submission.csv")

MODEL_NAME = "facebook/bart-large-mnli"
OUTPUT_DIR = Path("/kaggle/working/bart_large_mnli_metadata")
MAX_LEN = 256

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Sin GPU")
print("GPUs:", torch.cuda.device_count())


# DATA


train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

test_ids = test_df["id"].copy()

def build_model_text(df):
    df = df.copy()

    cols = [
        "statement",
        "subject",
        "speaker",
        "speaker_job",
        "state_info",
        "party_affiliation",
    ]

    for col in cols:
        df[col] = (
            df[col]
            .fillna("unknown")
            .astype(str)
            .str.strip()
        )

    df["model_text"] = (
        "Claim: " + df["statement"]
        + " Topic: " + df["subject"]
        + " Speaker: " + df["speaker"]
        + " Speaker job: " + df["speaker_job"]
        + " State: " + df["state_info"]
        + " Party: " + df["party_affiliation"]
    )

    return df

train_df = build_model_text(train_df)
test_df = build_model_text(test_df)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Label distribution:")
print(train_df["label"].value_counts())


# TOKENIZER / DATASET


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def encode(batch):
    return tokenizer(
        batch["model_text"],
        truncation=True,
        max_length=MAX_LEN,
    )

tr_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=SEED,
)

train_ds = Dataset.from_pandas(tr_df[["model_text", "label"]])
val_ds = Dataset.from_pandas(val_df[["model_text", "label"]])
test_ds = Dataset.from_pandas(test_df[["model_text"]])

train_ds = train_ds.map(encode, batched=True, remove_columns=["model_text"])
val_ds = val_ds.map(encode, batched=True, remove_columns=["model_text"])
test_ds = test_ds.map(encode, batched=True, remove_columns=["model_text"])

collator = DataCollatorWithPadding(
    tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)


# HELPERS


def extract_logits(predictions):
    if isinstance(predictions, tuple):
        return predictions[0]
    return predictions

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    logits = extract_logits(predictions)
    preds = np.argmax(logits, axis=-1)

    return {
        "f1": f1_score(labels, preds, average="macro")
    }


# MODEL


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)

model.config.use_cache = False
model.config.output_hidden_states = False
model.config.output_attentions = False
model.config.return_dict = True


# TRAINING ARGS


args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=8e-6,
    warmup_ratio=0.1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    eval_accumulation_steps=16,
    num_train_epochs=3,
    weight_decay=0.01,
    max_grad_norm=0.5,
    fp16=False,
    bf16=False,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=0,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    prediction_loss_only=False,
    remove_unused_columns=True,
)


# TRAINER


try:
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    )
except TypeError:
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    )

trainer.train()

metrics = trainer.evaluate()
print("F1 macro valid argmax:", metrics["eval_f1"])


# VALIDATION THRESHOLD


val_predictions = trainer.predict(val_ds).predictions
val_logits = extract_logits(val_predictions)
val_probs = softmax(val_logits, axis=1)[:, 1]

thresholds = np.arange(0.40, 0.76, 0.01)
scores = []

for threshold in thresholds:
    val_preds = (val_probs >= threshold).astype(int)
    score = f1_score(val_df["label"].values, val_preds, average="macro")
    scores.append(score)

best_idx = np.argmax(scores)
best_threshold = float(thresholds[best_idx])
best_f1 = float(scores[best_idx])

print("Mejor threshold:", best_threshold)
print("Mejor F1 macro valid:", best_f1)


# TEST PREDICTIONS


test_predictions = trainer.predict(test_ds).predictions
test_logits = extract_logits(test_predictions)
test_probs = softmax(test_logits, axis=1)[:, 1]


# SAVE PROBS FOR ENSEMBLE


bart_val_probs = pd.DataFrame({
    "id": val_df["id"].values,
    "label": val_df["label"].values,
    "bart_proba_1": val_probs,
})

bart_test_probs = pd.DataFrame({
    "id": test_ids,
    "bart_proba_1": test_probs,
})

bart_val_probs.to_csv("/kaggle/working/bart_val_probs.csv", index=False)
bart_test_probs.to_csv("/kaggle/working/bart_test_probs.csv", index=False)

print("Probabilidades guardadas:")
print("/kaggle/working/bart_val_probs.csv")
print("/kaggle/working/bart_test_probs.csv")


# SUBMISSIONS


thresholds_to_submit = sorted(set([
    round(best_threshold - 0.04, 2),
    round(best_threshold - 0.02, 2),
    round(best_threshold, 2),
    round(best_threshold + 0.02, 2),
    round(best_threshold + 0.04, 2),
    0.52,
    0.54,
    0.56,
    0.58,
    0.60,
    0.62,
    0.64,
]))

for threshold in thresholds_to_submit:
    if threshold < 0.05 or threshold > 0.95:
        continue

    preds = (test_probs >= threshold).astype(int)

    submission = sample_submission.copy()
    submission["label"] = submission["id"].map(dict(zip(test_ids, preds)))

    output_path = Path("/kaggle/working") / f"submission_bart_large_mnli_t{threshold:.2f}.csv"
    submission.to_csv(output_path, index=False)

    print("\nThreshold:", threshold)
    print("Archivo:", output_path)
    print(submission["label"].value_counts())


# CLEAN MEMORY


torch.cuda.empty_cache()
gc.collect()


CUDA: True
GPU: Tesla T4
GPUs: 1
Train: (8950, 9)
Test: (3836, 8)
Label distribution:
label
1    5795
0    3155
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/7160 [00:00<?, ? examples/s]

Map:   0%|          | 0/1790 [00:00<?, ? examples/s]

Map:   0%|          | 0/3836 [00:00<?, ? examples/s]

Some weights of BartForSequenceClassification were not initialized from the model checkpoint at facebook/bart-large-mnli and are newly initialized because the shapes did not match:
- classification_head.out_proj.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
- classification_head.out_proj.weight: found shape torch.Size([3, 1024]) in the checkpoint and torch.Size([2, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 

## 8. Stacking / Target Encoding con metadatos

Experimento para combinar texto TF-IDF, target encoding out-of-fold de variables contextuales y probabilidades OOF de modelos previos.


### 8.1 Carga de datos y revisin de probabilidades disponibles


In [15]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/kaggle/working")

DATA_DIR = Path("/kaggle/input/datasets/rosalirodriguez/testgpu-2")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test_nolabel.csv"
SAMPLE_PATH = DATA_DIR / "sample_submission.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)o
sample_submission = pd.read_csv(SAMPLE_PATH)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Sample:", sample_submission.shape)

print("\nLabel distribution:")
print(train_df["label"].value_counts())

print("\nArchivos de probabilidades disponibles:")
for p in ROOT.glob("*probs*.csv"):
    print(p.name)


Train: (8950, 8)
Test: (3836, 7)
Sample: (3836, 2)

Label distribution:
label
1    5795
0    3155
Name: count, dtype: int64

Archivos de probabilidades disponibles:
roberta_large_5fold_oof_probs.csv
roberta_large_5fold_test_probs.csv
electra_test_probs.csv
electra_val_probs.csv


### 8.2 Funciones auxiliares para target encoding OOF


In [19]:
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_SPLITS = 5

def normalize_cat(series):
    return (
        series
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .replace("", "unknown")
    )

def oof_target_encode(train, test, col, target="label", n_splits=5, smoothing=20):
    train = train.copy()
    test = test.copy()

    train[col] = normalize_cat(train[col])
    test[col] = normalize_cat(test[col])

    global_mean = train[target].mean()
    oof = np.zeros(len(train))
    
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=SEED
    )

    for tr_idx, val_idx in skf.split(train, train[target]):
        tr = train.iloc[tr_idx]
        val = train.iloc[val_idx]

        stats = tr.groupby(col)[target].agg(["mean", "count"])

        smooth = (
            (stats["mean"] * stats["count"] + global_mean * smoothing)
            / (stats["count"] + smoothing)
        )

        oof[val_idx] = val[col].map(smooth).fillna(global_mean)

    full_stats = train.groupby(col)[target].agg(["mean", "count"])

    full_smooth = (
        (full_stats["mean"] * full_stats["count"] + global_mean * smoothing)
        / (full_stats["count"] + smoothing)
    )

    test_encoded = test[col].map(full_smooth).fillna(global_mean)

    return oof, test_encoded.values


def count_encode(train, test, col):
    train = train.copy()
    test = test.copy()

    train[col] = normalize_cat(train[col])
    test[col] = normalize_cat(test[col])

    counts = train[col].value_counts()

    train_count = train[col].map(counts).fillna(0).values
    test_count = test[col].map(counts).fillna(0).values

    return np.log1p(train_count), np.log1p(test_count)


def prepare_subject_main(df):
    df = df.copy()
    df["subject_main"] = (
        df["subject"]
        .fillna("unknown")
        .astype(str)
        .str.split(",")
        .str[0]
        .str.strip()
        .replace("", "unknown")
    )
    return df


### 8.3 Construccion de features tabulares y textuales


In [20]:
train_fe = train_df.copy()
test_fe = test_df.copy()

train_fe = prepare_subject_main(train_fe)
test_fe = prepare_subject_main(test_fe)

cat_cols = [
    "speaker",
    "speaker_job",
    "state_info",
    "party_affiliation",
    "subject_main",
]

for col in cat_cols:
    train_fe[col] = normalize_cat(train_fe[col])
    test_fe[col] = normalize_cat(test_fe[col])

# Texto base
train_features = pd.DataFrame({
    "id": train_fe["id"],
    "statement": train_fe["statement"].fillna("").astype(str),
})

test_features = pd.DataFrame({
    "id": test_fe["id"],
    "statement": test_fe["statement"].fillna("").astype(str),
})

# Target encoding y count encoding
for col in cat_cols:
    te_train, te_test = oof_target_encode(
        train_fe,
        test_fe,
        col=col,
        target="label",
        n_splits=N_SPLITS,
        smoothing=20,
    )

    cnt_train, cnt_test = count_encode(train_fe, test_fe, col)

    train_features[f"{col}_te"] = te_train
    test_features[f"{col}_te"] = te_test

    train_features[f"{col}_count"] = cnt_train
    test_features[f"{col}_count"] = cnt_test

# Missing flags
train_features["speaker_job_missing"] = train_df["speaker_job"].isnull().astype(int)
test_features["speaker_job_missing"] = test_df["speaker_job"].isnull().astype(int)

train_features["state_info_missing"] = train_df["state_info"].isnull().astype(int)
test_features["state_info_missing"] = test_df["state_info"].isnull().astype(int)

# Subject count
train_features["subject_count"] = (
    train_df["subject"].fillna("").astype(str).str.split(",").apply(len)
)
test_features["subject_count"] = (
    test_df["subject"].fillna("").astype(str).str.split(",").apply(len)
)

y = train_df["label"].astype(int).values

print("Features iniciales train:", train_features.shape)
print("Features iniciales test:", test_features.shape)
display(train_features.head())


Features iniciales train: (8950, 15)
Features iniciales test: (3836, 15)


,id,statement,speaker_te,speaker_count,speaker_job_te,speaker_job_count,state_info_te,state_info_count,party_affiliation_te,party_affiliation_count,subject_main_te,subject_main_count,speaker_job_missing,state_info_missing,subject_count
0,81f884c64a7,China is in the South China Sea and (building)...,0.813408,5.513429,0.813408,5.513429,0.680480,6.364751,0.693410,8.280964,0.722334,3.637586,0,0,3
1,30c2723a188,With the resources it takes to execute just ov...,0.574989,2.079442,0.583910,5.468060,0.448743,3.091042,0.567833,7.972121,0.726672,6.480045,0,0,1
2,6936b216e5d,The (Wisconsin) governor has proposed tax give...,0.590730,2.302585,0.611281,2.772589,0.642344,4.605170,0.568122,7.972121,0.666089,4.477337,0,0,4
3,b5cd9195738,Says her representation of an ex-boyfriend who...,0.616653,1.098612,0.694601,7.817223,0.713526,7.565793,0.689753,7.334329,0.670365,6.269096,1,1,5
4,84f8dac7737,At protests in Wisconsin against proposed coll...,0.728835,2.397895,0.698674,7.817223,0.717999,6.476972,0.691711,8.280964,0.732485,6.480045,1,0,3


### 8.4 Carga de probabilidades OOF de modelos previos


In [22]:
def find_file(possible_names):
    for name in possible_names:
        path = ROOT / name
        if path.exists():
            return path
    return None

def add_oof_probs(
    train_features,
    test_features,
    model_name,
    oof_names,
    test_names,
):
    oof_path = find_file(oof_names)
    test_path = find_file(test_names)

    if oof_path is None or test_path is None:
        print(f"[SKIP] {model_name}: no encontré OOF o test probs.")
        return train_features, test_features

    oof_df = pd.read_csv(oof_path)
    test_df_probs = pd.read_csv(test_path)

    print(f"\n{model_name} OOF file:", oof_path.name)
    print(f"{model_name} TEST file:", test_path.name)
    print("OOF columns:", oof_df.columns.tolist())
    print("TEST columns:", test_df_probs.columns.tolist())

    if len(oof_df) != len(train_df):
        print(f"[SKIP] {model_name}: OOF tiene {len(oof_df)} filas, train tiene {len(train_df)}.")
        print("Esto parece valid probs, no OOF completo.")
        return train_features, test_features

    proba_cols_oof = [
        c for c in oof_df.columns
        if "proba" in c.lower() or "prob" in c.lower()
    ]

    proba_cols_test = [
        c for c in test_df_probs.columns
        if "proba" in c.lower() or "prob" in c.lower()
    ]

    if len(proba_cols_oof) == 0 or len(proba_cols_test) == 0:
        print(f"[SKIP] {model_name}: no encontré columna de probabilidad.")
        return train_features, test_features

    oof_proba_col = proba_cols_oof[0]
    test_proba_col = proba_cols_test[0]

    train_map = dict(zip(oof_df["id"], oof_df[oof_proba_col]))
    test_map = dict(zip(test_df_probs["id"], test_df_probs[test_proba_col]))

    train_features[f"{model_name}_proba_1"] = train_features["id"].map(train_map)
    test_features[f"{model_name}_proba_1"] = test_features["id"].map(test_map)

    missing_train = train_features[f"{model_name}_proba_1"].isnull().sum()
    missing_test = test_features[f"{model_name}_proba_1"].isnull().sum()

    print(f"{model_name} missing train:", missing_train)
    print(f"{model_name} missing test:", missing_test)

    if missing_train > 0 or missing_test > 0:
        print(f"[SKIP] {model_name}: hay ids sin probabilidad.")
        train_features = train_features.drop(columns=[f"{model_name}_proba_1"])
        test_features = test_features.drop(columns=[f"{model_name}_proba_1"])

    return train_features, test_features


# RoBERTa 5-fold OOF
train_features, test_features = add_oof_probs(
    train_features,
    test_features,
    model_name="roberta",
    oof_names=[
        "roberta_large_5fold_oof_probs.csv",
        "roberta_oof_probs.csv",
        "roberta_val_probs.csv",
    ],
    test_names=[
        "roberta_large_5fold_test_probs.csv",
        "roberta_test_probs.csv",
    ],
)

# ELECTRA 
train_features, test_features = add_oof_probs(
    train_features,
    test_features,
    model_name="electra",
    oof_names=[
        "electra_oof_probs.csv",
        "electra_large_oof_probs.csv",
        "electra_val_probs.csv",
    ],
    test_names=[
        "electra_test_probs.csv",
        "electra_large_test_probs.csv",
    ],
)

# BART para guardar probs
train_features, test_features = add_oof_probs(
    train_features,
    test_features,
    model_name="bart",
    oof_names=[
        "bart_oof_probs.csv",
        "bart_val_probs.csv",
    ],
    test_names=[
        "bart_test_probs.csv",
    ],
)

print("\nFinal train features:", train_features.shape)
print("Final test features:", test_features.shape)

print("\nColumnas usadas:")
print(train_features.columns.tolist())



roberta OOF file: roberta_large_5fold_oof_probs.csv
roberta TEST file: roberta_large_5fold_test_probs.csv
OOF columns: ['id', 'label', 'roberta_large_oof_proba_1']
TEST columns: ['id', 'roberta_large_test_proba_1']
roberta missing train: 0
roberta missing test: 0

electra OOF file: electra_val_probs.csv
electra TEST file: electra_test_probs.csv
OOF columns: ['id', 'label', 'electra_proba_1']
TEST columns: ['id', 'electra_proba_1']
[SKIP] electra: OOF tiene 1790 filas, train tiene 8950.
Esto parece valid probs, no OOF completo.
[SKIP] bart: no encontré OOF o test probs.

Final train features: (8950, 16)
Final test features: (3836, 16)

Columnas usadas:
['id', 'statement', 'speaker_te', 'speaker_count', 'speaker_job_te', 'speaker_job_count', 'state_info_te', 'state_info_count', 'party_affiliation_te', 'party_affiliation_count', 'subject_main_te', 'subject_main_count', 'speaker_job_missing', 'state_info_missing', 'subject_count', 'roberta_proba_1']


### 8.5 Entrenamiento del meta-modelo y generacion de submissions


In [23]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from scipy.special import expit

X = train_features.drop(columns=["id"])
X_test = test_features.drop(columns=["id"])

numeric_cols = [c for c in X.columns if c != "statement"]

print("Numeric columns:")
print(numeric_cols)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED,
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 3),
                min_df=2,
                max_df=0.90,
                max_features=60000,
                sublinear_tf=True,
            ),
            "statement",
        ),
        (
            "num",
            StandardScaler(),
            numeric_cols,
        ),
    ],
    remainder="drop",
)

best_model = None
best_c = None
best_valid_f1 = -1
best_valid_probs = None

for C in [0.05, 0.1, 0.3, 0.7, 1.0, 2.0]:
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    C=C,
                    max_iter=3000,
                    solver="saga",
                    class_weight="balanced",
                    n_jobs=-1,
                    random_state=SEED,
                ),
            ),
        ]
    )

    model.fit(X_train, y_train)

    valid_probs = model.predict_proba(X_valid)[:, 1]

    thresholds = np.arange(0.35, 0.76, 0.01)
    scores = []

    for threshold in thresholds:
        preds = (valid_probs >= threshold).astype(int)
        scores.append(f1_score(y_valid, preds, average="macro"))

    idx = np.argmax(scores)
    score = scores[idx]
    threshold = thresholds[idx]

    print(f"C={C} | best threshold={threshold:.2f} | valid F1 macro={score:.5f}")

    if score > best_valid_f1:
        best_valid_f1 = score
        best_c = C
        best_threshold = float(threshold)
        best_model = model
        best_valid_probs = valid_probs

print("\nBEST")
print("C:", best_c)
print("Threshold:", best_threshold)
print("Valid F1 macro:", best_valid_f1)

valid_preds = (best_valid_probs >= best_threshold).astype(int)

print("\nClassification report:")
print(classification_report(y_valid, valid_preds))

print("\nConfusion matrix:")
print(confusion_matrix(y_valid, valid_preds))


final_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                C=best_c,
                max_iter=3000,
                solver="saga",
                class_weight="balanced",
                n_jobs=-1,
                random_state=SEED,
            ),
        ),
    ]
)

final_model.fit(X, y)

test_probs = final_model.predict_proba(X_test)[:, 1]

thresholds_to_submit = sorted(set([
    round(best_threshold - 0.05, 2),
    round(best_threshold - 0.03, 2),
    round(best_threshold - 0.02, 2),
    round(best_threshold, 2),
    round(best_threshold + 0.02, 2),
    round(best_threshold + 0.03, 2),
    round(best_threshold + 0.05, 2),
    0.50,
    0.55,
    0.60,
    0.65,
]))

for threshold in thresholds_to_submit:
    if threshold < 0.05 or threshold > 0.95:
        continue

    preds = (test_probs >= threshold).astype(int)

    submission = sample_submission.copy()
    submission["label"] = submission["id"].map(dict(zip(test_features["id"], preds)))

    output_path = ROOT / f"submission_stacking_te_tfidf_t{threshold:.2f}.csv"
    submission.to_csv(output_path, index=False)

    print("\nThreshold:", threshold)
    print("Archivo:", output_path)
    print(submission["label"].value_counts())


Numeric columns:
['speaker_te', 'speaker_count', 'speaker_job_te', 'speaker_job_count', 'state_info_te', 'state_info_count', 'party_affiliation_te', 'party_affiliation_count', 'subject_main_te', 'subject_main_count', 'speaker_job_missing', 'state_info_missing', 'subject_count', 'roberta_proba_1']
C=0.05 | best threshold=0.49 | valid F1 macro=0.63903
C=0.1 | best threshold=0.48 | valid F1 macro=0.63822
C=0.3 | best threshold=0.46 | valid F1 macro=0.63635
C=0.7 | best threshold=0.41 | valid F1 macro=0.64355
C=1.0 | best threshold=0.41 | valid F1 macro=0.63665
C=2.0 | best threshold=0.48 | valid F1 macro=0.63667

BEST
C: 0.7
Threshold: 0.41000000000000003
Valid F1 macro: 0.6435515091301518

Classification report:
              precision    recall  f1-score   support

           0       0.57      0.47      0.52       631
           1       0.74      0.81      0.77      1159

    accuracy                           0.69      1790
   macro avg       0.65      0.64      0.64      1790
weighted

## 9. Utilidades de limpieza

Eliminar checkpoints o carpetas temporales pesadas generadas durante pruebas de folds.


In [ ]:
import shutil
from pathlib import Path

for p in Path("/kaggle/working").glob("roberta_large_fold_*"):
    shutil.rmtree(p, ignore_errors=True)
